# TrustCV Medical Deep Learning Showcase (Synthetic Cardiology)

**Notebook file name:** `TrustCV_Medical_DeepLearning_Showcase.ipynb`

## What this notebook demonstrates

Using a *single synthetic cardiology dataset* (no PHI), we show how **TrustCV** can help you evaluate models under more trustworthy splits:

1. **Binary classification** (1-year MACE) with **leakage detection** using `TrustCVValidator`.
2. **Per-fold outputs** using `UniversalCVRunner` (when data is simple `np.ndarray`).
3. **Manual CV with TrustCV splitters** for advanced deep-learning data types:
   - **Multi-label classification** (3 ECG/echo risk flags; sigmoid with C outputs)
   - **Regression** (future EF drop)
   - **Multi-input models** (clinical + imaging inputs; X as dict)
   - **Multi-output models** (classification + regression heads; y as dict)
   - **Custom training loop** (`train_step`) and **tf.data.Dataset** support

> **Expected pattern:**  
> - When you ignore patient groups, TrustCV’s leakage check should flag leakage.  
> - When you use grouped splits, leakage should pass and performance typically drops a bit (more realistic generalization).

---

## Clinical story (copy/paste into a paper or README)

We have patients with:
- **Tabular clinical variables:** age, BMI, systolic BP
- **Two imaging-derived biomarkers:** LV ejection fraction (EF, regression) and LV mass index (regression)
- **Three binary risk flags from ECG/echo:** AF present, LVH present, conduction abnormality (multi‑label, C=3)

**Outcomes**
- **Main task:** 1‑year MACE risk (**binary classification**)
- **Secondary task:** future EF drop (**regression**)

We simulate this dataset with *patient IDs* and *repeated visits per patient* to demonstrate why **group-aware CV** matters in medical ML.


In [ ]:
# (Optional) If running in a clean environment, uncomment:
# !pip -q install trustcv scikit-learn pandas matplotlib

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    mean_squared_error, r2_score
)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.multioutput import MultiOutputClassifier, MultiOutputRegressor
from sklearn.multiclass import OneVsRestClassifier

from trustcv import TrustCVValidator, UniversalCVRunner
from trustcv import StratifiedGroupKFold, GroupKFoldMedical, KFoldMedical

SEED = 42
np.random.seed(SEED)

import sklearn
print("sklearn version:", sklearn.__version__)



## 1) Generate a synthetic cardiology dataset with patient IDs and repeated visits

In [ ]:
def generate_synthetic_cardiology(n_patients=200, visits_per_patient=4, seed=42):
    rng = np.random.default_rng(seed)
    N = n_patients * visits_per_patient
    patient_ids = np.repeat(np.arange(n_patients), visits_per_patient).astype(int)

    # --- patient-level latent variables (stable across visits) ---
    age_p   = rng.normal(65, 10, n_patients)
    bmi_p   = rng.normal(27,  4, n_patients)
    sbp_p   = rng.normal(135, 15, n_patients)

    ef_p    = rng.normal(55,  8, n_patients)     # LV ejection fraction (%)
    lvm_p   = rng.normal(90, 20, n_patients)     # LV mass index (g/m2)

    # ECG/echo risk flags (patient tendency)
    af_p   = (rng.random(n_patients) < 0.25).astype(int)
    lvh_p  = (rng.random(n_patients) < 0.30).astype(int)
    cond_p = (rng.random(n_patients) < 0.20).astype(int)

    # --- visit-level noise ---
    age   = np.repeat(age_p, visits_per_patient) + rng.normal(0, 1.0, N)
    bmi   = np.repeat(bmi_p, visits_per_patient) + rng.normal(0, 0.6, N)
    sbp   = np.repeat(sbp_p, visits_per_patient) + rng.normal(0, 3.0, N)

    ef    = np.repeat(ef_p, visits_per_patient)  + rng.normal(0, 2.5, N)
    lvm   = np.repeat(lvm_p, visits_per_patient) + rng.normal(0, 6.0, N)

    # Multi-label flags inherit patient labels with small noise (some visits differ)
    flip_p = 0.08
    af   = np.repeat(af_p, visits_per_patient)
    lvh  = np.repeat(lvh_p, visits_per_patient)
    cond = np.repeat(cond_p, visits_per_patient)

    flips = (rng.random((N, 3)) < flip_p).astype(int)
    Y_multi = np.stack([af, lvh, cond], axis=1) ^ flips
    # ensure each visit has at least one label (optional)
    empty = np.where(Y_multi.sum(axis=1) == 0)[0]
    if len(empty) > 0:
        Y_multi[empty, rng.integers(0, 3, size=len(empty))] = 1

    af, lvh, cond = Y_multi[:, 0], Y_multi[:, 1], Y_multi[:, 2]

    # Main binary outcome: 1y MACE risk
    logit = (
        0.05*(age - 60)
        + 0.03*(sbp - 130)
        - 0.07*(ef  - 55)
        + 0.9*af + 0.6*lvh + 0.7*cond
        + rng.normal(0, 0.4, N)  # unobserved variability
    )
    prob = 1/(1+np.exp(-logit))
    y_main = (rng.random(N) < prob).astype(int)

    # Secondary regression: future EF drop (higher = worse)
    y_ef_drop = (
        np.maximum(0, 9 - 0.12*(ef - 55))
        + 2.0*af + 1.2*lvh + 1.5*cond
        + rng.normal(0, 1.8, N)
    ).astype("float32")

    # Features
    X_tab = np.stack([age, bmi, sbp], axis=1).astype("float32")
    X_img = np.stack([ef, lvm], axis=1).astype("float32")

    X_single = np.concatenate([X_tab, X_img], axis=1).astype("float32")  # easiest path
    X_dict   = {"clinical": X_tab, "imaging": X_img}                      # multi-input demo

    return X_single, X_tab, X_img, X_dict, Y_multi.astype(int), y_main.astype(int), y_ef_drop, patient_ids

X_single, X_tab, X_img, X_dict, y_multi, y_main, y_ef_drop, patient_ids = generate_synthetic_cardiology()

print("X_single:", X_single.shape, "y_main:", y_main.shape, "y_multi:", y_multi.shape, "y_ef_drop:", y_ef_drop.shape)
print("Unique patients:", len(np.unique(patient_ids)))

df = pd.DataFrame(X_single, columns=["age","bmi","sbp","ef","lvmass"])
df["patient_id"] = patient_ids
df["mace_1y"] = y_main
print(df.head())

print("\nGlobal label prevalence (multi-label flags):", np.round(y_multi.mean(axis=0), 3), "for [AF, LVH, COND]")
print("MACE prevalence:", round(y_main.mean(), 3))


## 2) TrustCVValidator: show why patient leakage matters (wrong split vs correct split)

In [ ]:
# Baseline sklearn model (fast + interpretable)
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])

# --- WRONG: StratifiedKFold ignores groups, so a patient can appear in both train/test folds ---
validator_bad = TrustCVValidator(
    method="stratified_kfold",
    n_splits=5,
    random_state=SEED,
    shuffle=True,
    check_leakage=True,
    check_balance=True,
)

res_bad = validator_bad.validate(model=baseline, X=X_single, y=y_main, groups=patient_ids)
print(res_bad.summary())
print("Leakage map:", res_bad.leakage_check)


In [ ]:
# --- CORRECT: grouped split prevents patient overlap between train/test folds ---
validator_good = TrustCVValidator(
    method="stratified_grouped_kfold",   # group-aware + stratified
    n_splits=5,
    random_state=SEED,
    shuffle=True,
    check_leakage=True,
    check_balance=True,
)

res_good = validator_good.validate(model=baseline, X=X_single, y=y_main, groups=patient_ids)
print(res_good.summary())
print("Leakage map:", res_good.leakage_check)


### What to look for (interpretation)

- In the **wrong** configuration, you should see **Leakage Check: FAILED** and a message like *group leakage detected*.
- In the **correct** configuration, leakage should **PASS**.
- Performance (e.g., AUC) often **drops a bit** under grouped CV—this is a *feature*, not a bug: it reflects generalization to unseen patients.


## 3) A lightweight Keras wrapper (only needed when you want sklearn-style CV APIs)

In [ ]:
# Lightweight sklearn helpers used in place of TensorFlow/Keras to keep the notebook portable.

def make_mlp_classifier(input_dim, hidden=(32, 16)):
    return MLPClassifier(
        hidden_layer_sizes=hidden,
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=64,
        learning_rate_init=1e-3,
        max_iter=200,
        random_state=SEED,
    )

def make_mlp_regressor(input_dim, hidden=(32, 16)):
    return MLPRegressor(
        hidden_layer_sizes=hidden,
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=64,
        learning_rate_init=1e-3,
        max_iter=300,
        random_state=SEED,
    )



## 4) UniversalCVRunner (array input): Deep learning binary MACE model on `X_single`

In [ ]:
mace_mlp = make_mlp_classifier(X_single.shape[1])

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)

runner = UniversalCVRunner(cv_splitter=cv, framework="sklearn", verbose=2)
results = runner.run(model=mace_mlp, data=(X_single, y_main, patient_ids))

# Post-hoc metrics (because sklearn adapter focuses on classification metrics for 1D labels)
fold_rows = []
for i, (_, val_idx) in enumerate(results.indices):
    y_true = y_main[val_idx]
    y_pred = results.predictions[i]
    proba = results.probabilities[i]
    y_score = proba[:, 1]
    fold_rows.append({
        "fold": i+1,
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "n_val": len(val_idx)
    })

pd.DataFrame(fold_rows)



### Interpretation

This section shows **per-fold** metrics for a Keras binary classifier under **StratifiedGroupKFold**:
- Fold metrics help diagnose instability (variance across folds).
- Grouped splitting ensures each fold contains unseen patients.


## 5) Manual CV with TrustCV splitters: Multi-input model (X as dict)

In [ ]:
def slice_X(X, idx):
    if isinstance(X, dict):
        return {k: v[idx] for k, v in X.items()}
    if isinstance(X, (list, tuple)):
        return [v[idx] for v in X]
    if hasattr(X, "iloc"):
        return X.iloc[idx]
    return X[idx]

def slice_y(y, idx):
    if isinstance(y, dict):
        return {k: v[idx] for k, v in y.items()}
    if hasattr(y, "iloc"):
        return y.iloc[idx]
    return y[idx]

def build_multi_input_mace():
    return make_mlp_classifier(input_dim=X_single.shape[1])

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)

rows = []
for fold, (tr, te) in enumerate(cv.split(X_single, y_main, patient_ids), 1):
    X_tr = np.concatenate([X_dict["clinical"][tr], X_dict["imaging"][tr]], axis=1)
    X_te = np.concatenate([X_dict["clinical"][te], X_dict["imaging"][te]], axis=1)
    y_tr, y_te = y_main[tr], y_main[te]

    model = build_multi_input_mace()
    model.fit(X_tr, y_tr)

    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)

    rows.append({
        "fold": fold,
        "accuracy": accuracy_score(y_te, pred),
        "roc_auc": roc_auc_score(y_te, proba),
        "n_val": len(te)
    })

pd.DataFrame(rows)



### Interpretation

- This demonstrates **multi-input Keras** (`X` is a dict: `{"clinical":..., "imaging":...}`).
- We still use TrustCV’s **group-aware splitter** to avoid patient leakage.
- This is a practical pattern today when your CV runner expects `np.ndarray` slicing but your data is dict/list.


## 6) Multi-label classification (sigmoid with C outputs): ECG/echo flags

In [ ]:
def build_multilabel_flags(input_dim=5, C=3):
    return OneVsRestClassifier(LogisticRegression(max_iter=1000))

cv = GroupKFoldMedical(n_splits=3)  # group separation is the key here.

rows = []
for fold, (tr, te) in enumerate(cv.split(X_single, y_multi, groups=patient_ids), 1):
    X_tr, X_te = X_single[tr], X_single[te]
    y_tr, y_te = y_multi[tr], y_multi[te]

    m = build_multilabel_flags(input_dim=X_single.shape[1], C=y_multi.shape[1])
    m.fit(X_tr, y_tr)

    proba = m.predict_proba(X_te)               # (n, C)
    pred  = (proba >= 0.5).astype(int)          # threshold

    rows.append({
        "fold": fold,
        "subset_acc": float(np.mean(np.all(pred == y_te, axis=1))),
        "f1_micro": f1_score(y_te, pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_te, pred, average="macro", zero_division=0),
        "n_val": len(te)
    })

pd.DataFrame(rows)



### Interpretation

- **Multi-label** means each sample can have **multiple 1s** (AF/LVH/COND).
- We report:
  - **subset accuracy** (exact match across all labels; very strict)
  - **micro-F1** (weights frequent labels more)
  - **macro-F1** (treats each label equally; highlights rare-label performance)


## 7) Regression: predict future EF drop (continuous y)

In [ ]:
def build_regressor(input_dim=5):
    return make_mlp_regressor(input_dim=input_dim)

cv = GroupKFoldMedical(n_splits=3)

rows = []
for fold, (tr, te) in enumerate(cv.split(X_single, y_ef_drop, groups=patient_ids), 1):
    X_tr, X_te = X_single[tr], X_single[te]
    y_tr, y_te = y_ef_drop[tr], y_ef_drop[te]

    m = build_regressor(input_dim=X_single.shape[1])
    m.fit(X_tr, y_tr)

    pred = m.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    rows.append({
        "fold": fold,
        "rmse": float(rmse),
        "r2": float(r2_score(y_te, pred)),
        "n_val": len(te)
    })

pd.DataFrame(rows)



### Interpretation

Regression CV outputs are typically summarized with **RMSE** (error scale) and **R²** (explained variance).
Because we simulated signal (EF + flags), the model should achieve a **moderate R²** (often ~0.3–0.7).


## 8) Multi-output model: classification + regression head (y as dict)

In [ ]:
def build_multioutput_models(input_dim=5):
    clf = LogisticRegression(max_iter=1000)
    reg = make_mlp_regressor(input_dim=input_dim)
    return clf, reg

y_dict = {"mace": y_main.astype("float32"), "ef_drop": y_ef_drop.astype("float32")}
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)

rows = []
for fold, (tr, te) in enumerate(cv.split(X_single, y_main, patient_ids), 1):
    X_tr, X_te = X_single[tr], X_single[te]
    y_tr, y_te = slice_y(y_dict, tr), slice_y(y_dict, te)

    clf, reg = build_multioutput_models(input_dim=X_single.shape[1])
    clf.fit(X_tr, y_tr["mace"].astype(int))
    reg.fit(X_tr, y_tr["ef_drop"])

    p_mace = clf.predict_proba(X_te)[:, 1]
    pred_mace = clf.predict(X_te)

    pred_drop = reg.predict(X_te)

    rows.append({
        "fold": fold,
        "mace_auc": roc_auc_score(y_te["mace"], p_mace),
        "mace_acc": accuracy_score(y_te["mace"].astype(int), pred_mace),
        "ef_rmse": float(np.sqrt(mean_squared_error(y_te["ef_drop"], pred_drop))),
        "ef_r2": float(r2_score(y_te["ef_drop"], pred_drop)),
        "n_val": len(te)
    })

pd.DataFrame(rows)



### Interpretation

This shows **multi-task learning**:
- A shared representation feeds both a **binary risk head** and a **regression head**.
- You typically evaluate each head with task-appropriate metrics:
  - classification: **AUC**, accuracy
  - regression: **RMSE**, R²


## 9) Custom training loop (train_step) + tf.data.Dataset + extra data (sample weights)

In [ ]:
class CustomBinaryModel:
    """Simple custom estimator that supports sample weights for demonstration."""
    def __init__(self):
        self.model = LogisticRegression(max_iter=1000)

    def fit(self, X, y, sample_weight=None):
        self.model.fit(X, y, sample_weight=sample_weight)
        return self

    def predict(self, X):
        return self.model.predict(X)

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def score(self, X, y, sample_weight=None):
        return self.model.score(X, y, sample_weight=sample_weight)

# Extra data object: sample weights emphasize high-risk positives
# Here: weight positives 2x, negatives 1x (typical medical trick to fight imbalance)
sample_weight = np.where(y_main == 1, 2.0, 1.0).astype("float32")

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)

rows = []
for fold, (tr, te) in enumerate(cv.split(X_single, y_main, patient_ids), 1):
    X_tr, X_te = X_single[tr], X_single[te]
    y_tr, y_te = y_main[tr].astype(int), y_main[te].astype(int)
    sw_tr = sample_weight[tr]

    m = CustomBinaryModel()
    m.fit(X_tr, y_tr, sample_weight=sw_tr)

    p = m.predict_proba(X_te)[:, 1]
    pred = (p >= 0.5).astype(int)

    rows.append({
        "fold": fold,
        "auc": roc_auc_score(y_te, p),
        "acc": accuracy_score(y_te, pred),
        "n_val": len(te)
    })

pd.DataFrame(rows)



### Interpretation

This section demonstrates:
- **Custom `train_step`** (useful for advanced losses, multi-stage objectives, or special logging)
- Training via **tf.data.Dataset**
- Passing **sample weights** as an extra data object

These are common “real-life” medical DL requirements and show TrustCV splitters still work with them.


## Summary: what this notebook proves

- **TrustCVValidator** is ideal when your data is array-like and you want:
  - a single call to get a **summary report** (means, std, CIs)
  - built-in **leakage checks** and balance checks

- **UniversalCVRunner** is ideal when you want:
  - **per-fold models**, **per-fold predictions**, callbacks, and a unified runner

- For **dict/list inputs**, **multi-output**, and **tf.data/custom loops**, the most reliable pattern today is:
  1) use TrustCV splitters to generate indices  
  2) slice your complex X/y manually  
  3) train/evaluate Keras models per fold

If you want, I can also generate a “companion notebook” that focuses purely on **multilabel+group stratification** (iterative stratification with groups), because that is a special case that needs a dedicated splitter.
